[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/text-classification-practice/02_keras_text/02_keras_text.ipynb)

# 02. 같은 문제를 딥러닝으로 — Keras 텍스트 분류

[01번](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/text-classification-practice/01_text_baseline/01_text_baseline.ipynb)에서
TF-IDF + 로지스틱 회귀로 검증 정확도 **0.8455** 를 얻었습니다. 이 노트북은 같은 데이터를
신경망으로 다시 풉니다.

## 이 장을 배우는 이유

목표는 두 가지입니다.

1. **텍스트를 신경망에 넣는 방법**을 익힌다 — 정수 시퀀스, 패딩, [임베딩](https://github.com/karzit/temp/blob/master/glossary.md#embedding)
2. **딥러닝이 항상 이기지는 않는다**는 것을 확인하고, **왜 지는지를 숫자로 밝힌다**

두 번째가 이 노트북의 진짜 주제입니다. "신경망이 졌다"에서 멈추면 다음에 무엇을 해야 할지 알 수 없습니다.
**진 이유를 찾아내면, 그 이유를 고쳐서 얼마나 되찾을 수 있는지도 잴 수 있습니다.**
실제로 이 노트북에서 그렇게 합니다.

## TF-IDF와 임베딩은 무엇이 다른가

| | TF-IDF (01번) | 임베딩 (이 노트북) |
|---|---|---|
| 입력 표현 | 사전 크기만큼의 **희소 벡터** (대부분 0) | 단어마다 **길이 64짜리 조밀 벡터** |
| 벡터 값 | 통계로 계산 (등장 횟수 × IDF) | **학습으로 결정** (역전파로 갱신) |
| 어순 | 사라짐 | `Conv1D`/`LSTM`을 쓰면 **일부 반영** |
| 단어 사이 관계 | 없음 (`코스피`와 `코스닥`은 남남) | 비슷한 문맥의 단어가 **가까운 벡터**가 됨 |
| 사전에 없는 단어 | 그냥 무시됨 | 전부 `[UNK]` 한 칸으로 뭉개짐 |
| 데이터가 적을 때 | 강함 | 약함 (배울 것이 많아 과적합) |

**임베딩의 값은 학습으로 정해집니다.** 이것이 핵심 차이입니다. 대신 정해야 할 값이 훨씬 많아서,
데이터가 적으면 그 자유도가 그대로 [과적합](https://github.com/karzit/temp/blob/master/glossary.md#overfitting)이 됩니다.
표의 다섯째 줄(사전에 없는 단어)도 기억해두세요. **5절에서 이 줄이 승부를 가릅니다.**

## 이 노트북의 구성

| 절 | 내용 |
|---|---|
| 1~2 | 텍스트 → 정수 시퀀스 → 패딩, 라벨 인코딩 |
| 3 | 가장 단순한 모델 (임베딩 + 평균) |
| 4 | `Conv1D`, `LSTM`과 비교 |
| 5 | 01번의 TF-IDF 모델과 정면 비교 — **그리고 진 이유 찾기** |
| 6 | 찾아낸 원인을 고쳐본다 — 글자 단위 |
| 7 | **모델 저장의 함정** — 저장은 됐는데 예측이 안 된다 |
| 8 | 새 데이터에서 최종 확인 |

> **소요 시간 60분쯤.** 학습 셀은 CPU에서 모델 하나에 20~30초입니다(4절에서 세 개를 연달아 학습합니다).
>
> **선수 지식.** Keras의 `Sequential`·`compile`·`fit`·`EarlyStopping`을 처음 본다면
> [tabular-ml-practice 04번](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/tabular-ml-practice/04_dnn_keras/04_dnn_keras.ipynb)을
> 먼저 보세요. 여기서는 **텍스트에 특수한 부분**만 설명합니다.

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    !pip install -q pandas scikit-learn matplotlib koreanize-matplotlib

YNAT = "https://raw.githubusercontent.com/KLUE-benchmark/KLUE/main/klue_benchmark/ynat-v1.1"

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

try:
    import koreanize_matplotlib  # noqa: F401
except ImportError:
    pass

RANDOM_STATE = 42
keras.utils.set_random_seed(RANDOM_STATE)  # 재현성을 위해 seed를 고정합니다

print("TensorFlow", tf.__version__, "· Keras", keras.__version__)

**01번과 똑같이 데이터를 준비합니다.** 같은 20,000건, 같은 `random_state`로 나눕니다.
그래야 5절의 비교가 공정합니다. 조건을 하나라도 다르게 두면 "신경망이 졌다"는 결론이
데이터 분할 탓인지 모델 탓인지 구분할 수 없습니다.

In [ ]:
from sklearn.model_selection import train_test_split

raw = pd.read_json(f"{YNAT}/ynat-v1.1_train.json")
data = raw[["title", "label"]].sample(20_000, random_state=RANDOM_STATE).reset_index(drop=True)

X = data["title"].values
y_text = data["label"].values

X_train, X_valid, y_train_text, y_valid_text = train_test_split(
    X, y_text, test_size=0.2, stratify=y_text, random_state=RANDOM_STATE
)
print("학습", len(X_train), "· 검증", len(X_valid))

---

## 1. 텍스트를 정수 시퀀스로

신경망은 TF-IDF처럼 "문서 하나 = 벡터 하나"로 받지 않습니다. **단어를 순서대로 늘어놓은
정수 배열**로 받습니다.

```
"남북정상 핫라인 열렸다"  →  [812, 4507, 6]  →  [812, 4507, 6, 0, 0, 0, ...]
                              정수로 바꾸고       길이를 맞춰 0으로 채움(패딩)
```

`TextVectorization` 레이어가 이 두 가지를 한 번에 합니다.

| 인자 | 뜻 | 정하는 법 |
|---|---|---|
| `max_tokens` | 사전에 담을 단어 수 | 자주 나오는 단어부터. 넘치면 `[UNK]` 처리 |
| `output_sequence_length` | 시퀀스 길이 | **단어 수 분포를 보고** 정함 |
| `standardize` | 소문자화·문장부호 제거 | 기본값이 이미 둘 다 함 |

**중요:** `adapt()`는 **학습 데이터에만** 부릅니다. 전체 데이터로 사전을 만들면
01번에서 본 [데이터 누출](https://github.com/karzit/temp/blob/master/glossary.md#data-leakage)입니다.

먼저 시퀀스 길이부터 정합니다.

In [ ]:
단어수 = pd.Series([len(s.split()) for s in X_train])
print(단어수.describe().round(1))
print("\n95% 지점:", int(단어수.quantile(0.95)), "단어 · 99% 지점:", int(단어수.quantile(0.99)), "단어")

평균 6.6단어이고 95%가 9단어 이하, 최대가 13단어입니다. 길이를 **12**로 잡으면 거의 자르지 않습니다.

**시퀀스 길이를 정하는 기준**은 간단합니다. 너무 짧으면 뒤가 잘려 정보를 잃고,
너무 길면 대부분이 0(패딩)이라 계산만 낭비합니다. **분위수를 보고 95~99% 지점**으로 잡으면 무난합니다.

다음은 사전 크기입니다. 이쪽은 그렇게 간단하지 않습니다.

In [ ]:
# max_tokens를 주지 않으면 학습 데이터에 나온 단어를 전부 사전에 담습니다.
전체사전 = layers.TextVectorization()
전체사전.adapt(X_train)
print("제한 없이 만든 사전 크기:", len(전체사전.get_vocabulary()), "단어")

# 그중 몇 개가 딱 한 번만 나오는 단어일까요?
from collections import Counter

토큰 = [w for s in X_train for w in s.lower().split()]
빈도 = Counter(토큰)
print("전체 토큰 %d개 · 서로 다른 단어 %d개" % (len(토큰), len(빈도)))
print("딱 한 번만 나온 단어의 비율: %.1f%%" % (sum(1 for n in 빈도.values() if n == 1) / len(빈도) * 100))

**제목 16,000건에서 서로 다른 단어가 44,901개 나왔습니다. 그중 74%가 딱 한 번만 등장합니다.**

01번 6절에서 사전이 40,429개였던 것과 같은 현상입니다. 원인도 같습니다 — **조사와 어미**입니다.
`정부는`, `정부가`, `정부의`가 전부 다른 단어로 세어집니다.

한 번만 나온 단어는 신경망에 아무 쓸모가 없습니다. **한 번 본 것으로는 그 단어의 벡터를
학습할 수 없기 때문입니다.** 그렇다고 사전에서 빼면 그 자리는 `[UNK]`가 됩니다.
어느 쪽이든 손해인 상황이고, `max_tokens`는 **이 손해를 어디서 끊을지** 정하는 값입니다.

20,000으로 잡고, **그 대가가 얼마인지 재봅니다.**

In [ ]:
MAX_TOKENS = 20_000
SEQ_LEN = 12

vectorize = layers.TextVectorization(
    max_tokens=MAX_TOKENS,
    output_sequence_length=SEQ_LEN,
)
vectorize.adapt(X_train)   # 학습 데이터로만 사전을 만듭니다

vocab = vectorize.get_vocabulary()
print("사전 크기:", len(vocab))
print("앞 10개:", vocab[:10])

# 검증 데이터를 정수로 바꿨을 때, 패딩이 아닌 자리 중 몇 %가 [UNK](=1)인가?
ids = vectorize(X_valid).numpy()
print("\n검증 데이터의 OOV 비율: %.1f%%" % ((ids == 1).sum() / (ids != 0).sum() * 100))

print("\n예시:", X_train[0])
print("→", vectorize([X_train[0]]).numpy()[0])

**검증 데이터에 들어 있는 단어의 38.5%가 `[UNK]`입니다.**

이 숫자를 기억해두세요. **모델이 제목을 읽을 때 단어 열 개 중 넷은 "모르는 단어"라는 뜻입니다.**
5절에서 이 숫자가 결정적인 역할을 합니다.

출력된 예시도 확인하세요. `국방부 北 풍계리 핵실험장 계속 감시중…준비는 완료돼`가
`[758, 3, 4707, 4632, 323, 1, 1, 0, ...]`이 됐습니다. **`감시중`과 `준비는`이 둘 다 `1`, 즉 `[UNK]`입니다.**
흔한 단어인데도 그렇습니다 — `감시`와 `준비`는 사전에 있지만 `감시중`, `준비는`은 없기 때문입니다.

사전의 **0번은 패딩(빈 자리), 1번은 `[UNK]`** 로 예약되어 있고, 뒤쪽이 0으로 채워진 것이 패딩입니다.

---

## 2. 라벨을 숫자로

주제는 문자열입니다. 신경망 출력층은 숫자만 압니다.
`LabelEncoder`로 `가나다순 → 0~6`으로 바꿉니다.

In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_train_text)   # 학습 데이터로 fit
y_valid = label_encoder.transform(y_valid_text)

N_CLASSES = len(label_encoder.classes_)
print(N_CLASSES, "개 주제")
for i, c in enumerate(label_encoder.classes_):
    print(f"  {i}: {c}")

**손실 함수는 `sparse_categorical_crossentropy`를 씁니다.** 라벨이 `[0, 3, 6, ...]`처럼
정수 하나로 되어 있을 때 쓰는 것이고, [원-핫](https://github.com/karzit/temp/blob/master/glossary.md#one-hot-encoding)으로
바꿔뒀다면 `categorical_crossentropy`입니다. 둘은 수학적으로 같고 **입력 형태만 다릅니다.**
원-핫으로 바꾸는 단계가 없으니 이쪽이 간단합니다.

`label_encoder.classes_`의 순서가 곧 출력층의 순서입니다. **예측값을 다시 문자열로 되돌릴 때
같은 인코더를 써야** 하므로, 7절에서 모델과 함께 저장합니다.

---

## 3. 모델 만들기 — 임베딩 + 평균

첫 모델은 최대한 단순하게 갑니다.

```
문자열 → TextVectorization → Embedding → GlobalAveragePooling1D → Dense → softmax
```

- **`Embedding(사전 크기, 64)`**: 단어 하나를 길이 64짜리 벡터로 바꿉니다.
  이 벡터가 [역전파](https://github.com/karzit/temp/blob/master/glossary.md#backpropagation)로 학습됩니다
- **`GlobalAveragePooling1D`**: 단어 벡터 12개를 **평균 내어 하나로** 만듭니다.
  어순은 무시됩니다. 사실상 "학습되는 BoW"입니다. 길이가 다른 시퀀스를 **고정 길이 벡터 하나로
  줄이는 단계**가 반드시 필요한데, 평균이 그중 가장 단순한 방법입니다
  (뒤에 나오는 `GlobalMaxPooling1D`는 평균 대신 **각 자리의 최댓값**을 취합니다.
  "가장 강하게 반응한 신호만 남긴다"는 뜻이라, 핵심어 하나를 찾는 문제에 잘 맞습니다)
- **[`Dropout`](https://github.com/karzit/temp/blob/master/glossary.md#dropout)(0.3)**: 학습 중에 뉴런의 30%를 무작위로 끕니다. 과적합 대책입니다
- **출력층 `Dense(7, activation="softmax")`**: 주제 7개에 대한 확률

### `Sequential`이 아니라 함수형 API를 씁니다

`tabular-ml-practice` 04번에서는 층을 리스트로 쌓는 `Sequential`을 썼습니다. 여기서는 조금 다른
방식으로 씁니다.

```python
# Sequential — 층을 순서대로 나열
model = keras.Sequential([layers.Dense(64), layers.Dense(7)])

# 함수형 API — 입력을 만들고, 층을 함수처럼 호출해 이어 붙인다
inputs = keras.Input(shape=(1,), dtype=tf.string)   # 문자열 한 칸짜리 입력
x = vectorize(inputs)                                # 층(inputs) 형태로 통과시킴
outputs = layers.Dense(7, activation="softmax")(x)
model = keras.Model(inputs, outputs)                 # 입구와 출구를 지정해 모델 완성
```

**둘은 같은 모델을 만듭니다.** 함수형 쪽은 "입력이 어떤 자료형인지"(`dtype=tf.string`)를 지정할 수 있고,
`vectorize`처럼 **이미 만들어둔 레이어를 중간에 끼워 넣기** 편해서 여기서 씁니다.
`x = 층(x)`는 "x를 이 층에 통과시킨 결과를 다시 x라고 부른다"는 뜻일 뿐입니다.

`TextVectorization`을 **모델 안에 넣었다**는 점을 기억해두세요. 그러면 모델이 문자열을 그대로 받으므로
예측할 때 전처리를 다시 재현할 필요가 없습니다. (7절에서 이 선택의 대가를 봅니다.)

In [ ]:
inputs = keras.Input(shape=(1,), dtype=tf.string)   # 문자열 한 칸이 입력
x = vectorize(inputs)                                # → 정수 12개
x = layers.Embedding(MAX_TOKENS, 64, name="embedding")(x)   # → 12 × 64 벡터
x = layers.GlobalAveragePooling1D()(x)               # → 64 벡터 하나로
x = layers.Dropout(0.3)(x)
x = layers.Dense(64, activation="relu")(x)
outputs = layers.Dense(N_CLASSES, activation="softmax")(x)

model = keras.Model(inputs, outputs)
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()

**파라미터 수를 보세요. 1,284,615개 중 1,280,000개가 `Embedding` 층에 있습니다**
(사전 20,000 × 64차원). 학습 데이터는 16,000건인데 학습할 값은 128만 개입니다.

**데이터 한 건당 파라미터 80개꼴입니다.** 표 데이터에서라면 말이 안 되는 비율입니다.
`Dropout`과 `EarlyStopping`이 선택이 아니라 필수인 이유이고, 1절에서 본 "74%가 한 번만 나오는 단어"가
왜 문제인지도 여기서 다시 보입니다 — **한 번밖에 안 나온 단어의 64개짜리 벡터를
어떻게 제대로 학습하겠습니까.**

이제 학습합니다. `epochs=30`으로 넉넉히 잡되, **검증 손실이 3 epoch 동안 나아지지 않으면
`EarlyStopping`이 멈춥니다.** 30번을 다 돌지 않고 10 언저리에서 멈추면 정상입니다.

In [ ]:
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=3, restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    validation_data=(X_valid, y_valid),
    epochs=30,
    batch_size=64,
    callbacks=[early_stop],
    verbose=0,
)

hist = pd.DataFrame(history.history)   # epoch별 loss/accuracy 기록
print("학습한 epoch 수:", len(hist))
print("검증 정확도 최고: %.4f" % hist["val_accuracy"].max())

**검증 정확도 0.75쯤에서 멈췄고, 7 epoch 만에 `EarlyStopping`이 걸렸습니다.**
30번을 다 돌지 않았다는 것은 **일찌감치 과적합이 시작됐다**는 뜻입니다. 그림으로 확인합니다.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
hist[["loss", "val_loss"]].plot(ax=axes[0], title="손실")
hist[["accuracy", "val_accuracy"]].plot(ax=axes[1], title="정확도")
axes[0].set_xlabel("epoch")
axes[1].set_xlabel("epoch")
plt.tight_layout()
plt.show()

**학습 곡선을 읽으세요.** 학습 손실은 계속 내려가는데 검증 손실이 어느 시점부터 올라갑니다.
그 지점이 과적합의 시작이고, `EarlyStopping(restore_best_weights=True)`이 그 시점의 가중치를 되돌려줍니다.

두 곡선이 **일찍, 크게 벌어지는 것**이 이 모델의 특징입니다. 파라미터 128만 개로 16,000건을
학습하니 당연한 결과입니다.

---

## 4. 구조 세 가지 비교

`GlobalAveragePooling1D`은 어순을 버립니다. 어순을 보는 구조를 두 개 더 시도합니다.

| head | 하는 일 | 어순 |
|---|---|---|
| `average` | 단어 벡터를 평균 | 무시 |
| `conv` | **이웃한 3단어 묶음**에서 패턴을 찾음 (`Conv1D`) | 지역적으로 반영 |
| `lstm` | 앞에서 뒤로(그리고 뒤에서 앞으로) 읽으며 상태를 유지 | 전체 반영 |

[`LSTM`](https://github.com/karzit/temp/blob/master/glossary.md#lstm-gru)은 [RNN](https://github.com/karzit/temp/blob/master/glossary.md#rnn)의 개선판으로,
단어를 하나씩 읽으면서 **지금까지 읽은 내용을 상태로 들고 갑니다**(원리는
[ml-curriculum 06번](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/ml-curriculum/06_rnn/06_rnn.ipynb)에서 다룹니다).
`Bidirectional`로 감싸면 **앞→뒤와 뒤→앞을 둘 다** 읽어 두 결과를 이어 붙입니다.

[CNN](https://github.com/karzit/temp/blob/master/glossary.md#cnn)을 이미지가 아니라 **텍스트에 1차원으로** 쓰는 것이 `Conv1D`입니다.
이미지에서 3×3 필터가 인접 픽셀을 보듯, 여기서는 인접한 3개 단어를 봅니다.

**뉴스 제목은 단어를 나열한 것이 아니라 어순이 있는 문장입니다.** "북한이 미국을 비판"과
"미국이 북한을 비판"은 뜻이 다릅니다. 그러니 어순을 보는 구조가 유리해야 합니다.
**정말 그런지 확인합시다.**

세 번 비슷한 코드를 쓰지 않도록, 3절에서 만든 모델을 **함수로 묶고 가운데 한 부분만 바꿉니다.**

In [ ]:
def build_model(head="average"):
    """head: average | conv | lstm — 가운데 부분만 바꿔 모델을 만든다."""
    inputs = keras.Input(shape=(1,), dtype=tf.string)
    x = vectorize(inputs)
    x = layers.Embedding(MAX_TOKENS, 64, name="embedding")(x)

    # ↓ 여기만 다르다
    if head == "average":
        x = layers.GlobalAveragePooling1D()(x)          # 평균 (3절과 동일)
    elif head == "conv":
        x = layers.Conv1D(128, 3, activation="relu")(x)  # 이웃 3단어 묶음에서 패턴 찾기
        x = layers.GlobalMaxPooling1D()(x)               # 가장 강한 신호만 남기기
    elif head == "lstm":
        x = layers.Bidirectional(layers.LSTM(64))(x)     # 앞뒤로 읽으며 상태 유지
    # ↑ 여기까지

    x = layers.Dropout(0.3)(x)
    x = layers.Dense(64, activation="relu")(x)
    outputs = layers.Dense(N_CLASSES, activation="softmax")(x)

    model = keras.Model(inputs, outputs)
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

세 구조를 같은 조건(같은 seed, 같은 `EarlyStopping`)으로 학습해 나란히 비교합니다.
**모델 하나에 20~30초**, 셋을 합쳐 1분 30초쯤 걸립니다.

In [ ]:
import time

results = {}
for head in ["average", "conv", "lstm"]:
    keras.utils.set_random_seed(RANDOM_STATE)
    started = time.time()

    m = build_model(head)
    h = m.fit(
        X_train, y_train,
        validation_data=(X_valid, y_valid),
        epochs=30, batch_size=64,
        callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=3,
                                                 restore_best_weights=True)],
        verbose=0,
    )
    # evaluate는 compile에 넣은 순서대로 [손실, 정확도]를 돌려줍니다. [1]이 정확도입니다.
    val_acc = m.evaluate(X_valid, y_valid, verbose=0)[1]
    results[head] = (m, val_acc)
    print(f"{head:<8} 검증 정확도 {val_acc:.4f}  ({len(h.history['loss'])} epoch, {time.time() - started:.0f}초)")

**어순을 보는 구조가 이기지 못했습니다.**

| head | 정확도 |
|---|---|
| average (어순 무시) | **0.7523** |
| conv (이웃 3단어) | 0.7487 |
| lstm (전체 어순) | 0.7300 |

**가장 단순한 `average`가 가장 좋고, 가장 정교한 `lstm`이 가장 나쁩니다.**
"뉴스 제목에는 어순이 있으니 LSTM이 유리할 것"이라는 예상이 빗나갔습니다.

이유는 어순이 없어서가 아닙니다. **어순을 볼 형편이 아니어서**입니다.

- `LSTM`은 파라미터가 더 많은데, 이미 데이터 한 건당 파라미터 80개인 상황입니다.
  자유도를 더 주면 과적합만 심해집니다
- 그리고 1절을 떠올리세요. **입력 단어의 38.5%가 `[UNK]`입니다.**
  `[UNK] [UNK] 열렸다`에서 어순을 읽어봐야 얻을 것이 없습니다

**입력이 부실하면 구조를 정교하게 만들어도 소용없습니다.** 이것이 다음 절의 주제입니다.

---

## 5. 01번의 TF-IDF 모델과 정면 비교

같은 데이터, 같은 분할로 01번의 모델을 다시 학습해 나란히 놓습니다.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.pipeline import make_pipeline

tfidf_model = make_pipeline(
    TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 3)),
    LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
).fit(X_train, y_train_text)

tfidf_acc = accuracy_score(y_valid_text, tfidf_model.predict(X_valid))

print(f"{'TF-IDF + 로지스틱 회귀':<24} {tfidf_acc:.4f}")
for head, (m, acc) in results.items():
    print(f"{'신경망 (' + head + ')':<24} {acc:.4f}")

**0.8455 대 0.7523. 9%p 차이로 졌습니다.**

01번의 선형 모델이 신경망을 크게 앞섭니다. 대충 진 것이 아니라 **완패**입니다.
`tabular-ml-practice` 04번에서 신경망이 랜덤 포레스트를 못 이겼던 것과 같은 장면인데,
격차는 훨씬 큽니다.

여기서 "역시 데이터가 적을 때는 고전 모델이 낫다"로 마무리하면 배울 것이 없습니다.
**두 모델이 정확히 무엇을 다르게 했는지** 짚어봅시다.

| | TF-IDF (0.8455) | 신경망 (0.7523) |
|---|---|---|
| 텍스트를 자르는 단위 | **글자 2~3개 조각** | **공백으로 나눈 단어** |
| 사전에 없는 것 | 애초에 없음 (조각은 다 만들어짐) | **입력의 38.5%가 `[UNK]`** |
| 학습할 파라미터 | 없음 (통계만 셈) | 128만 개 |
| 학습 데이터 | 16,000건 | 16,000건 |

**결정적인 줄은 두 번째입니다.** 신경망은 제목을 읽을 때 단어 열 개 중 넷을 "모르는 단어"로
처리하고 있었습니다. TF-IDF는 그런 일이 없습니다. `감시중`이라는 단어를 처음 보더라도
`감시`, `시중` 같은 **조각은 이미 알고 있기** 때문입니다.

**즉, 진 이유는 "신경망이라서"가 아니라 "단어 단위로 잘라서"입니다.**
01번 8절에서 문자 n-gram이 +8%p를 만들었던 것과 정확히 같은 이야기가,
이번에는 신경망 쪽에서 **잃는 방향으로** 나타났을 뿐입니다.

가설이 섰으면 확인해야 합니다.

---

## 6. 원인을 고쳐본다 — 글자 단위로 자르기

`TextVectorization`에 **`split="character"`** 를 주면 공백이 아니라 **글자 하나하나**로 자릅니다.

```
"남북정상 핫라인 열렸다"  →  ['남','북','정','상',' ','핫','라','인', ...]
```

이러면 사전이 **한글 음절 1,500개 남짓**으로 줄어듭니다(단어 단위였을 때는 20,000). 조사가 붙든 말든 글자는 그대로이므로
**`[UNK]`가 사실상 사라집니다.**

대신 시퀀스가 길어집니다. 12단어 대신 **40글자**를 처리해야 합니다(01번에서 제목 길이가
95% 지점에서 34글자였습니다).

그리고 이번에는 **`conv`** 를 씁니다. `Conv1D(128, 3)`이 인접한 **글자 3개 묶음**을 보게 되는데,
이것은 사실상 **학습되는 문자 3-gram**입니다. TF-IDF가 통계로 세던 것을 신경망이 학습으로 하는 셈입니다.

**아래 두 셀을 합쳐 30초쯤 걸립니다.**

In [ ]:
def build_char_model(head="conv", seq_len=40, max_tokens=3000):
    """글자 단위로 자르는 모델. split='character'만 다르다."""
    keras.utils.set_random_seed(RANDOM_STATE)

    char_vectorize = layers.TextVectorization(
        max_tokens=max_tokens,
        output_sequence_length=seq_len,
        split="character",          # 이 한 줄이 핵심
    )
    char_vectorize.adapt(X_train)

    inputs = keras.Input(shape=(1,), dtype=tf.string)
    x = char_vectorize(inputs)
    x = layers.Embedding(max_tokens, 64)(x)
    if head == "conv":
        x = layers.Conv1D(128, 3, activation="relu")(x)   # 인접 글자 3개 = 학습되는 문자 3-gram
        x = layers.GlobalMaxPooling1D()(x)
    else:
        x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(64, activation="relu")(x)
    outputs = layers.Dense(N_CLASSES, activation="softmax")(x)

    m = keras.Model(inputs, outputs)
    m.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m, char_vectorize


char_model, char_vectorize = build_char_model()
print("글자 사전 크기:", len(char_vectorize.get_vocabulary()), "(단어 단위였을 때는 20,000)")

In [ ]:
char_model.fit(
    X_train, y_train,
    validation_data=(X_valid, y_valid),
    epochs=30, batch_size=64, verbose=0,
    callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=3,
                                             restore_best_weights=True)],
)
char_acc = char_model.evaluate(X_valid, y_valid, verbose=0)[1]

print("신경망 · 단어 단위 (conv)  %.4f" % results["conv"][1])
print("신경망 · 글자 단위 (conv)  %.4f" % char_acc)
print("TF-IDF · 문자 n-gram      %.4f" % tfidf_acc)

**0.7487 → 0.7928. 한 줄을 바꿔 4.4%p를 되찾았습니다.**

가설이 맞았습니다. **문제는 모델 구조가 아니라 텍스트를 자르는 단위였습니다.**
`LSTM`을 붙이거나 층을 쌓아서는 얻을 수 없었던 향상을, `split="character"` 한 줄이 만들었습니다.

**그래도 TF-IDF(0.8455)에는 여전히 5%p 못 미칩니다.** 남은 격차의 이유는 이렇습니다.

- **데이터 양.** 임베딩 파라미터를 제대로 학습하려면 16,000건으로는 부족합니다.
  TF-IDF는 학습할 파라미터가 없고 통계만 세므로 적은 데이터에서 유리합니다
- **`Conv1D`는 인접한 3글자만** 봅니다. TF-IDF의 `char_wb` 2~3-gram은
  **낱말 경계를 인식**해서 조각을 만듭니다. 후자가 이 문제에 더 잘 맞는 피처였습니다

**그럼 딥러닝은 언제 쓰나요?** 텍스트가 길고 데이터가 많을 때, 그리고 **사전 학습 모델**을 쓸 때입니다.
KLUE-RoBERTa 같은 한국어 사전 학습 모델은 이미 대량의 한국어로 훈련되어 있어, 임베딩을 처음부터
학습할 필요가 없습니다. 사실 KLUE-YNAT는 **그런 모델들을 평가하려고 만들어진 데이터셋**입니다.
우리가 지금 한 것은 그 벤치마크를 맨손으로 풀어본 셈입니다.

> **이 절의 진짜 교훈.** "졌다"에서 멈추지 않고 **원인을 가설로 세우고 한 줄로 검증**했습니다.
> 성능이 안 나올 때 층을 더 쌓거나 epoch를 늘리는 것은 대개 시간 낭비입니다.
> **입력이 모델에게 어떻게 보이는지 먼저 확인하세요.** 1절에서 OOV 비율을 재둔 것이
> 여기서 값어치를 했습니다.

---

## 7. 모델 저장의 함정

학습한 모델을 파일로 남겨야 다음에 다시 씁니다. Keras에서 가장 오래된 방식은 `.h5`입니다.
저장해보겠습니다.

In [ ]:
import os

char_model.save("temp_model.h5")     # 저장은 됩니다 (경고가 나올 수 있습니다)
print("저장 완료:", round(os.path.getsize("temp_model.h5") / 1024), "KB")

reloaded = keras.models.load_model("temp_model.h5")
print("불러오기도 성공했습니다.")

try:
    reloaded.predict(X_valid[:3], verbose=0)
    print("예측 성공")
except Exception as e:
    print("\n예측 실패:", type(e).__name__)
    print(str(e).split("\n")[0])

**저장도 되고 불러오기도 되는데, 예측에서 터집니다.**

이것이 이 함정의 고약한 점입니다. `save()`와 `load_model()`이 아무 소리 없이 성공하기 때문에,
**실제로 써보기 전까지는 모델이 망가진 것을 모릅니다.**

원인은 `TextVectorization`입니다. 이 레이어가 들고 있는 것은 가중치가 아니라 **글자 사전(문자열 표)** 인데,
`.h5`는 숫자 배열만 담는 옛 포맷이라 그 사전을 복원하지 못합니다. 불러온 모델은 껍데기만 있고
"글자를 정수로 바꾸는 표"가 비어 있어서, 예측하려는 순간 실패합니다.

**해결책은 두 가지입니다.**

| 방법 | 하는 일 | 파일 |
|---|---|---|
| **A. `.keras`로 저장** | 최신 Keras 포맷은 전처리 레이어를 그대로 담는다 | `.keras` 하나 |
| **B. 벡터화를 모델 밖으로** | 모델은 정수 배열만 받게 하고, 사전은 따로 저장 | `.h5` + 사전 파일 |

**A가 정답입니다.** 특별한 이유가 없다면 `.keras`를 쓰세요.
B는 `.h5`를 꼭 써야 하는 사정이 있을 때의 방법입니다. 둘 다 해봅니다.

In [ ]:
# 방법 A — .keras 포맷
char_model.save("temp_model.keras")
reloaded_a = keras.models.load_model("temp_model.keras")

same = np.allclose(char_model.predict(X_valid[:20], verbose=0),
                   reloaded_a.predict(X_valid[:20], verbose=0))
print("A) .keras 불러오기 성공 · 예측이 원본과 동일:", same)

이번엔 방법 B입니다. **문제의 원인은 `TextVectorization`이 모델 안에 있다는 것**이었으니,
그 레이어를 **모델 밖으로 빼면** `.h5`도 쓸 수 있습니다. 모델은 문자열 대신 **정수 배열**을 받게 되고,
그만큼 예측할 때 벡터화를 직접 해줘야 합니다. 두 단계로 나눠 진행합니다 — 먼저 학습과 모델 저장입니다.

In [ ]:
# 방법 B-1 — 벡터화를 모델 밖에서 미리 해두고, 모델은 정수 배열만 받게 만든다
X_train_ids = char_vectorize(X_train).numpy()
X_valid_ids = char_vectorize(X_valid).numpy()

keras.utils.set_random_seed(RANDOM_STATE)
int_model = keras.Sequential([
    keras.Input(shape=(40,)),        # 문자열이 아니라 정수 40개를 받는다
    layers.Embedding(3000, 64),
    layers.Conv1D(128, 3, activation="relu"),
    layers.GlobalMaxPooling1D(),
    layers.Dropout(0.3),
    layers.Dense(64, activation="relu"),
    layers.Dense(N_CLASSES, activation="softmax"),
])
int_model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
int_model.fit(X_train_ids, y_train, validation_data=(X_valid_ids, y_valid),
              epochs=30, batch_size=64, verbose=0,
              callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=3,
                                                       restore_best_weights=True)])

int_model.save("temp_int_model.h5")
print("저장 완료. 이제 사전을 따로 저장합니다.")

모델은 저장했지만 **사전은 아직 저장되지 않았습니다.** 사전은 가중치(숫자)가 아니라 문자열 목록이라
모델 파일에 담기지 않습니다. 사전이 없으면 `"남북정상 핫라인 열렸다"`를 어떤 정수로 바꿔야 할지 알 수 없어,
불러온 모델이 아무 쓸모가 없습니다. **따로 저장합니다.**

In [ ]:
# 방법 B-2 — 사전을 json으로 저장하고, 모델과 사전을 함께 되살린다
import json

char_vocab = char_vectorize.get_vocabulary()
# 0번 패딩, 1번 [UNK]는 TextVectorization이 자동으로 만드므로 목록에서 제외합니다.
json.dump([str(w) for w in char_vocab[2:]],
          open("temp_vocab.json", "w", encoding="utf-8"), ensure_ascii=False)

reloaded_b = keras.models.load_model("temp_int_model.h5")
restored_vectorize = layers.TextVectorization(
    max_tokens=3000, output_sequence_length=40, split="character",
    vocabulary=json.load(open("temp_vocab.json", encoding="utf-8")),   # 저장해둔 사전을 주입
)

정확도 = (reloaded_b.predict(restored_vectorize(X_valid), verbose=0).argmax(axis=1) == y_valid).mean()
print("B) .h5 불러오기 성공 · 검증 정확도 %.4f" % 정확도)

**B의 교훈:** 모델 파일에 들어가지 않는 것(사전, 라벨 인코더, 스케일러)은 **따로 저장해야 합니다.**
표 데이터에서 "모델을 저장할 때 스케일러도 함께"였던 것과 같은 이야기입니다.

그리고 더 중요한 것.

> **저장하는 순간이 아니라, 불러와서 예측이 되는지 확인하는 순간까지가 저장입니다.**

`.h5` 사례가 보여주듯 `save()`와 `load_model()`이 성공했다는 것은 아무것도 보장하지 않습니다.
**저장 코드 바로 아래에 불러와서 예측해보는 코드를 붙여두세요.**

---

## 8. 새 데이터에서 최종 확인

01번 12절에서 TF-IDF 모델을 **KLUE의 별도 평가 데이터 9,107건**에 넣어봤습니다.
0.8455에서 0.7639로 떨어졌고, 원인은 `사회`의 비율이 11%에서 41%로 뛴 **분포 이동**이었습니다.

신경망은 어떨까요. 저장했던 `.keras` 모델을 그대로 씁니다.

In [ ]:
dev = pd.read_json(f"{YNAT}/ynat-v1.1_dev.json")[["title", "label"]]

nn_dev = (reloaded_a.predict(dev["title"].values, verbose=0).argmax(axis=1)
          == label_encoder.transform(dev["label"])).mean()
tfidf_dev = accuracy_score(dev["label"], tfidf_model.predict(dev["title"]))

print("               검증(같은 분포)   평가 데이터(다른 분포)")
print("신경망(글자)      %.4f            %.4f" % (char_acc, nn_dev))
print("TF-IDF          %.4f            %.4f" % (tfidf_acc, tfidf_dev))

**둘 다 떨어졌고, 순서는 그대로입니다.**

| | 검증 | 평가 데이터 | 하락 |
|---|---|---|---|
| 신경망 (글자 단위) | 0.7928 | 0.6913 | −0.10 |
| TF-IDF | 0.8455 | 0.7639 | −0.08 |

분포가 바뀌자 **신경망이 조금 더 크게 무너졌습니다.** 학습 데이터에 맞춰 임베딩을 처음부터
학습한 모델이라, 학습 때 본 분포에 더 단단히 묶여 있었던 것으로 보입니다.

**이 시리즈에서 배운 것을 한 줄로 요약하면 이렇습니다.**

> 짧은 한국어 텍스트를 분류할 때는, **문자 단위 TF-IDF + 선형 모델**로 먼저 튼튼한 기준선을 세우세요.
> 신경망은 그것을 이길 때만 값어치가 있고, 이 규모에서는 이기지 못했습니다.

**신경망을 쓸 이유가 없다는 뜻은 아닙니다.** 데이터가 10배 많거나, 텍스트가 문단 길이거나,
사전 학습 모델을 가져다 쓸 수 있다면 결론이 뒤집힙니다. 중요한 것은 **그 결론을 추측이 아니라
측정으로 내렸다는 점**입니다.

---

## 정리

- **텍스트를 신경망에 넣으려면** 정수 시퀀스 + 패딩이 필요합니다. `TextVectorization`이 둘 다 합니다
- **시퀀스 길이는 단어 수 분포의 95~99% 지점**으로 잡습니다
- **`adapt()`는 학습 데이터에만.** 사전을 전체 데이터로 만들면 데이터 누출입니다
- **사전을 만들었으면 OOV 비율을 재보세요.** 한국어 단어 단위에서 38.5%가 `[UNK]`였고,
  그것이 이 노트북에서 벌어진 모든 일의 원인이었습니다
- **임베딩은 학습되는 단어 표현**입니다. 대신 파라미터가 폭증하므로 `Dropout`·`EarlyStopping`이 필수입니다
- **구조(average/conv/lstm)를 바꿔서 얻은 것은 없었습니다.** 가장 단순한 `average`가 가장 좋았고,
  `LSTM`이 가장 나빴습니다. **입력이 부실하면 구조로 만회할 수 없습니다**
- **토큰 단위를 바꾸자 +4.4%p.** 구조를 바꿔서 얻지 못한 것을 `split="character"` 한 줄이 만들었습니다
- **딥러닝이 항상 이기지 않습니다.** 16,000건, 27글자짜리 텍스트에서는 TF-IDF + 선형 모델이 5~9%p 앞섭니다
- **저장은 불러와서 예측까지 해봐야 끝납니다.** `TextVectorization`을 품은 모델을 `.h5`로 저장하면
  **저장도 되고 불러와지지만 예측에서 실패합니다** — `.keras`를 쓰거나 사전을 따로 저장하세요

## 스스로 확인해보기

- [ ] `output_sequence_length`를 정하는 기준을 설명할 수 있다
- [ ] 사전의 0번과 1번이 무엇인지 안다
- [ ] 한국어를 공백으로 자르면 왜 OOV가 폭증하는지 설명할 수 있다
- [ ] `sparse_categorical_crossentropy`와 `categorical_crossentropy`의 차이를 안다
- [ ] `Conv1D`가 글자 단위 입력에서 무엇을 하는지 설명할 수 있다
- [ ] 이 문제에서 신경망이 TF-IDF를 못 이긴 이유를 세 가지 댈 수 있다
- [ ] `.h5` 저장이 실패하는 방식과 그 해결책 두 가지를 안다

## 연습 문제

풀어본 뒤 [02_keras_text_solutions.ipynb](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/text-classification-practice/02_keras_text/02_keras_text_solutions.ipynb)에서 확인하세요.

**문제 1.** `SEQ_LEN`을 4로 줄이면(단어 단위, `average`) 성능이 어떻게 되나요?
제목의 앞 4단어만 남는다는 뜻입니다. 01번 연습 문제 6번에서 "정보가 앞에 있는지 뒤에 있는지"를
확인했는데, 그 결과와 일치하나요?

**문제 2.** `MAX_TOKENS`를 5,000 / 20,000 / 44,901(전체 어휘)로 바꿔가며 **OOV 비율, 정확도,
파라미터 수**를 함께 비교하세요. 사전을 키우면 OOV가 줄어드는데, 정확도도 그만큼 올라가나요?
결과를 보고 "OOV를 줄이는 것"이 목표가 될 수 있는지 판단하세요.

**문제 3.** 임베딩 차원을 16 / 64 / 256으로 바꿔가며 검증 정확도와 학습 시간을 비교하세요.
차원을 키우면 항상 좋아지나요? 비용은 얼마나 드나요?

**문제 4.** 단어 단위 모델의 `Embedding` 층 가중치를 꺼내, `손흥민`과 코사인 유사도가 높은 단어
8개를 찾아보세요. 의미가 비슷한 단어가 실제로 가까운가요? 여기서 "의미"란 무엇을 뜻하나요?

**문제 5.** 신경망(글자 단위)과 TF-IDF의 예측 확률을 **평균 내어**(앙상블) 정확도를 재보세요.
앙상블은 이득인가요? 두 모델이 각각 어디서 틀리는지 세어보고 그 이유를 설명하세요.

**문제 6.** 학습 데이터를 45,678건 전체로 늘리면 신경망이 TF-IDF를 따라잡나요?
6절에서 "남은 격차의 이유는 데이터 양"이라고 했는데, 그 주장이 맞는지 확인하는 문제입니다.
(글자 단위 `conv` 기준으로 학습에 3분쯤 걸립니다.)

---

여기까지가 이 시리즈입니다. 더 나아가고 싶다면
[시리즈 README](https://github.com/karzit/temp/blob/master/notebooks/text-classification-practice/README.md)의
"다음으로 해볼 만한 것"을 보세요 — 사전 학습 한국어 모델(KLUE-RoBERTa), 형태소 분석기,
제목 대신 본문으로 넓히기로 이어집니다.